# 🍲 FoodShareAI — 2-Class Food Freshness Model Trainer & Verification

### CELL 1: Imports

In [ ]:
import os
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

print(f"TensorFlow Version: {tf.__version__}")

### CELL 2: Project paths and configuration

In [ ]:
from pathlib import Path

# Resolve project root dynamically
current_path = Path(os.getcwd()).resolve()
project_root = None
for parent in [current_path] + list(current_path.parents):
    if (parent / "freshness_dataset").exists():
        project_root = parent
        break
if project_root is None:
    if current_path.name == "scripts":
        project_root = current_path.parent
    else:
        project_root = current_path

DATASET_PATH = (project_root / "freshness_dataset").resolve()
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print(f"DATASET_PATH = {DATASET_PATH}")

### CELL 3: Dataset validation

In [ ]:
fresh_dir = DATASET_PATH / "Fresh"
spoiled_dir = DATASET_PATH / "Spoiled"

if not fresh_dir.exists() or not spoiled_dir.exists():
    raise FileNotFoundError(f"Missing directories inside resolved dataset path: {DATASET_PATH}")

# Count actual files
fresh_images = list(fresh_dir.glob("*.jpg")) + list(fresh_dir.glob("*.jpeg")) + list(fresh_dir.glob("*.png")) + list(fresh_dir.glob("*.webp"))
spoiled_images = list(spoiled_dir.glob("*.jpg")) + list(spoiled_dir.glob("*.jpeg")) + list(spoiled_dir.glob("*.png")) + list(spoiled_dir.glob("*.webp"))

FRESH_IMAGES = len(fresh_images)
SPOILED_IMAGES = len(spoiled_images)
TOTAL_IMAGES = FRESH_IMAGES + SPOILED_IMAGES

print(f"FRESH = {FRESH_IMAGES}")
print(f"SPOILED = {SPOILED_IMAGES}")
print(f"TOTAL = {TOTAL_IMAGES}")

if FRESH_IMAGES == 0 or SPOILED_IMAGES == 0:
    raise ValueError("Aborting: One of the class directories has zero images.")

### CELL 4: Load train/validation datasets

In [ ]:
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATASET_PATH),
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATASET_PATH),
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

CLASS_NAMES = raw_train_ds.class_names
print(f"CLASS_NAMES = {CLASS_NAMES}")

if len(CLASS_NAMES) != 2:
    raise ValueError(f"Expected exactly 2 classes, but got {len(CLASS_NAMES)}")

TRAINING_SAMPLES = len(raw_train_ds.file_paths)
VALIDATION_SAMPLES = len(raw_val_ds.file_paths)
print(f"TRAINING_SAMPLES = {TRAINING_SAMPLES}")
print(f"VALIDATION_SAMPLES = {VALIDATION_SAMPLES}")

### CELL 5: Dataset performance/prefetch

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Data Augmentation Sequentials (applied ONLY to training set)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="data_augmentation")

train_ds = raw_train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = raw_val_ds.prefetch(AUTOTUNE)

### CELL 6: Class weights

In [ ]:
total_samples = FRESH_IMAGES + SPOILED_IMAGES
class_weight_0 = total_samples / (2.0 * FRESH_IMAGES)
class_weight_1 = total_samples / (2.0 * SPOILED_IMAGES)
class_weights = {0: class_weight_0, 1: class_weight_1}
print("CLASS_WEIGHTS =", class_weights)

### CELL 7: Model creation

In [ ]:
def create_model():
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False

    data_augmentation_layer = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.10),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ])

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x = data_augmentation_layer(inputs)

    # Preprocessing is inside the model internally
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    x = base_model(x, training=False)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(
        2,
        activation="softmax",
        name="freshness_output"
    )(x)

    model = tf.keras.Model(
        inputs,
        outputs,
        name="FoodShareAI_Freshness_Model"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    return model

model = create_model()

### CELL 8: Model summary

In [ ]:
model.summary()
print("Model output shape:", model.output_shape)

### CELL 9: Training

In [ ]:
# To train the model, un-comment the lines below.
# By default, we keep the verified freshness_model_best.keras and do not automatically retrain it.

# callbacks = [
#     tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
#     tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6)
# ]
# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=15,
#     class_weight=class_weights,
#     callbacks=callbacks
# )
# model.save("freshness_model_best.keras")
# print("Saved best model to freshness_model_best.keras")

### CELL 10: Evaluation

In [ ]:
# Load the pre-trained best model
if os.path.exists("freshness_model_best.keras"):
    model = tf.keras.models.load_model("freshness_model_best.keras")
    print("Loaded existing freshness_model_best.keras for evaluation.")
else:
    print("WARNING: freshness_model_best.keras not found. Running evaluation on current un-trained/in-memory model.")

val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f"VALIDATION_ACCURACY = {val_accuracy:.4f}")
print(f"VALIDATION_LOSS = {val_loss:.4f}")

### CELL 11: Classification report

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(predictions, axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

### CELL 12: Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("FoodShareAI Freshness Confusion Matrix")
plt.tight_layout()
plt.show()

### CELL 13: Independent test prediction

In [ ]:
def predict_food_freshness(image_path):
    if not os.path.exists(image_path):
        print(f"File {image_path} does not exist.")
        return None, None, None
        
    image = tf.keras.utils.load_img(
        image_path,
        target_size=IMG_SIZE,
        color_mode="rgb"
    )

    image_array = tf.keras.utils.img_to_array(image)
    image_array = image_array.astype(np.float32)

    batch = np.expand_dims(image_array, axis=0)

    # Note: no external preprocess_input is applied, the model preprocesses internally
    probabilities = model.predict(batch, verbose=0)[0]

    predicted_index = int(np.argmax(probabilities))
    predicted_label = CLASS_NAMES[predicted_index]
    confidence = float(probabilities[predicted_index])

    print("Image:", image_path)
    print("Fresh probability:", float(probabilities[0]))
    print("Spoiled probability:", float(probabilities[1]))
    print("Predicted:", predicted_label)
    print("Confidence:", confidence)

    return predicted_label, confidence, probabilities

# Test on local verification images
test_dir = project_root / "test_freshness"
predict_food_freshness(str(test_dir / "fresh.jpg"))
predict_food_freshness(str(test_dir / "spoiled.jpg"))

### CELL 14: TFLite conversion/verification

In [ ]:
# To convert to TFLite, un-comment the lines below.
# By default, we keep the verified food_freshness.tflite and do not automatically overwrite it.

# print("Converting model to TFLite...")
# run_model = tf.function(lambda x: model(x))
# concrete_func = run_model.get_concrete_function(
#     tf.TensorSpec([1, 224, 224, 3], tf.float32)
# )
# converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
# tflite_model = converter.convert()
# with open("freshness_model_2class.tflite", "wb") as f:
#     f.write(tflite_model)
# print("Saved freshness_model_2class.tflite")